In [1]:
import requests

# Test URL for HDFC Top 100 Direct
url = "https://api.mfapi.in/mf/125497"
response = requests.get(url)

print(f"Response Status Code: {response.status_code}") # Should print 200

Response Status Code: 200


In [2]:
import pandas as pd
import os

# The 6 codes assigned to you by Bluestock
target_schemes = {
    "125497": "HDFC_Top_100_Direct",
    "119551": "SBI_Bluechip",
    "120503": "ICICI_Bluechip",
    "118632": "Nippon_Large_Cap",
    "119092": "Axis_Bluechip",
    "120841": "Kotak_Bluechip"
}

all_records = []

# Loop through each fund code and fetch data
for code, name in target_schemes.items():
    api_url = f"https://api.mfapi.in/mf/{code}"
    print(f"Fetching data for: {name}...")
    
    response = requests.get(api_url)
    if response.status_code == 200:
        json_data = response.json()
        
        # Grab the underlying data and metadata
        meta = json_data.get('meta', {})
        prices = json_data.get('data', [])
        
        # Pull out the date and NAV price for each entry
        for item in prices:
            all_records.append({
                "amfi_code": code,
                "scheme_name": meta.get('scheme_name'),
                "date": item.get('date'),
                "nav": item.get('nav')
            })

# Convert our list of data into a clean table (DataFrame)
live_df = pd.DataFrame(all_records)

# Look at the first 5 rows inside your notebook!
live_df.head()

Fetching data for: HDFC_Top_100_Direct...
Fetching data for: SBI_Bluechip...
Fetching data for: ICICI_Bluechip...
Fetching data for: Nippon_Large_Cap...
Fetching data for: Axis_Bluechip...
Fetching data for: Kotak_Bluechip...


,amfi_code,scheme_name,date,nav
0,125497,SBI Small Cap Fund - Direct Plan - Growth,02-06-2026,193.12950
1,125497,SBI Small Cap Fund - Direct Plan - Growth,01-06-2026,192.31950
2,125497,SBI Small Cap Fund - Direct Plan - Growth,31-05-2026,193.68360
3,125497,SBI Small Cap Fund - Direct Plan - Growth,29-05-2026,193.68480
4,125497,SBI Small Cap Fund - Direct Plan - Growth,27-05-2026,195.05010


In [3]:
# Create the raw data file pathway
output_path = os.path.join("..", "data-raw", "live_fetched_nav_data.csv")

# Save the table to that folder
live_df.to_csv(output_path, index=False)
print(f"✅ Success! Data safely stored in data-raw/live_fetched_nav_data.csv")

✅ Success! Data safely stored in data-raw/live_fetched_nav_data.csv


In [4]:
import os
import pandas as pd

# Define paths to your specific files
master_file = os.path.join("..", "data-raw", "01_fund_master.csv")
history_file = os.path.join("..", "data-raw", "02_nav_history.csv") # Check your actual filename if it's different!

print("--- Data Validation ---")
if os.path.exists(master_file):
    df_master = pd.read_csv(master_file)
    
    # 1. Print unique fund values as requested
    print(f"🏠 Unique Fund Houses: {df_master['fund_house'].nunique()}")
    print(f"⚠️ Unique Risk Categories: {df_master['risk_category'].unique()}")
    print(f"📋 Unique Main Categories: {df_master['category'].unique()}\n")
    
    # 2. Check if history file exists to cross-validate AMFI codes
    if os.path.exists(history_file):
        df_history = pd.read_csv(history_file)
        
        master_codes = set(df_master['amfi_code'].unique())
        history_codes = set(df_history['amfi_code'].unique()) # Adjust column name if it differs
        
        missing_codes = master_codes - history_codes
        print(f"🔢 Total unique AMFI codes in Master File: {len(master_codes)}")
        print(f"🔢 Total unique AMFI codes in History File: {len(history_codes)}")
        print(f"❌ AMFI codes present in Master but completely missing from History: {len(missing_codes)}")
else:
    print("⚠️ Please make sure '01_fund_master.csv' is inside your data-raw folder.")

--- Data Validation ---
🏠 Unique Fund Houses: 10
⚠️ Unique Risk Categories: ['Moderate' 'Very High' 'Low' 'High' 'Moderately High']
📋 Unique Main Categories: ['Equity' 'Debt']

🔢 Total unique AMFI codes in Master File: 40
🔢 Total unique AMFI codes in History File: 40
❌ AMFI codes present in Master but completely missing from History: 0
